# QQA 12 – Natural-language optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuma-Ichikawa/QQA4CO/blob/main/examples/12_natural_language_optimization_colab.ipynb)

One safe entry point for QQA, QQA+SCIP, one-run Pareto fronts, and budget-aware black-box optimization.

In [ ]:
# Install QQA on Google Colab (no-op if already installed locally).
# We prefer the released wheel on PyPI; users who want bleeding-edge
# ``main`` can set QQA_INSTALL_FROM_GIT=1 before running this cell.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("qqa") is None:
    if os.environ.get("QQA_INSTALL_FROM_GIT") == "1":
        spec = "qqa @ git+https://github.com/Yuma-Ichikawa/QQA4CO.git"
    else:
        spec = "qqa"
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", spec]
    )

## Why this notebook?

The same `qqa.ask(...)` call compiles an ordinary-language decision problem into a strict, reviewable model and routes it locally. The LLM never executes code or chooses an arbitrary solver. Keep API keys in an environment variable or a hidden prompt—never in the notebook.

In [ ]:
import getpass
import os

import qqa

print("QQA version:", qqa.__version__)
if not os.environ.get("QQA_LLM_API_KEY"):
    key = getpass.getpass("Compatible API key (leave blank for offline cells): ")
    if key:
        os.environ["QQA_LLM_API_KEY"] = key
if os.environ.get("QQA_LLM_API_KEY") and not os.environ.get("QQA_LLM_BASE_URL"):
    base_url = input("OpenAI-compatible base URL: ").strip()
    if base_url:
        os.environ["QQA_LLM_BASE_URL"] = base_url
if os.environ.get("QQA_LLM_API_KEY") and not os.environ.get("QQA_LLM_MODEL"):
    model_id = input("Model ID: ").strip()
    if model_id:
        os.environ["QQA_LLM_MODEL"] = model_id
live_api_ready = all(
    os.environ.get(name)
    for name in ("QQA_LLM_API_KEY", "QQA_LLM_BASE_URL", "QQA_LLM_MODEL")
)
print("Live API profile ready:", live_api_ready)

## 1. Review and solve without an API

A validated JSON model is ideal for reproducible production runs. This mixed binary/integer/real example works without credentials.

In [ ]:
production_spec = {
    "name": "production-plan",
    "variables": [
        {"name": "open", "kind": "binary", "lower": 0, "upper": 1, "size": 2},
        {"name": "lots", "kind": "integer", "lower": 0, "upper": 12, "size": 2},
        {"name": "overtime", "kind": "real", "lower": 0, "upper": 16, "size": 1},
    ],
    "objectives": [
        {
            "name": "weekly_cost",
            "direction": "min",
            "expression": "1400*open[0] + 1100*open[1] + 460*lots[0] + 510*lots[1] + 38*square(overtime)",
            "unit": "USD",
        }
    ],
    "constraints": [
        {
            "name": "demand", "expression": "8*lots[0] + 7*lots[1] + overtime",
            "sense": ">=", "rhs": 105, "weight": 1000, "scale": 105, "tolerance": 0.05,
        },
        {
            "name": "link_a", "expression": "lots[0] - 12*open[0]",
            "sense": "<=", "rhs": 0, "weight": 500, "scale": 12, "tolerance": 0.01,
        },
        {
            "name": "link_b", "expression": "lots[1] - 10*open[1]",
            "sense": "<=", "rhs": 0, "weight": 500, "scale": 10, "tolerance": 0.01,
        },
    ],
    "notes": "",
}
plan = qqa.plan_spec(production_spec, solver="qqa")
plan.to_dict()["routing"]

In [ ]:
answer = qqa.execute_plan(
    plan, sol_size=128, num_epochs=800, device="auto", seed=7
)
answer.result.score

## 2. Natural language → QQA + SCIP

With `qqa[scip]` installed, `solver="auto"` uses QQA exploration followed by SCIP certification for compatible single-objective models.

In [ ]:
single_request = """
Plan production at two plants. Opening decisions are binary, production
lots are bounded integers, and overtime is continuous. Minimize fixed,
lot, and quadratic overtime costs while meeting demand and linking each
plant's production to its opening decision. Use the numerical bounds and
coefficients from the reviewed production example above.
"""
if live_api_ready:
    single = qqa.ask(
        single_request,
        solver="auto",
        device="auto",
        sol_size=128,
        num_epochs=800,
        scip_time_limit=30,
    )
    display(single.plan.to_dict()["routing"])
    display(single.result.score)
else:
    print("Set the QQA_LLM_* API profile to run this live translation.")

## 3. Natural language → one-run Pareto front

Multiple objectives are preserved as separate goals. Parallel reference directions recover a nondominated archive in one run.

In [ ]:
pareto_request = """
Allocate integer production lots and continuous overtime while deciding
which plants open. Simultaneously minimize total cost, carbon emissions,
and unmet-demand risk. Keep each objective separate and enforce capacity,
activation, and demand constraints. Give every variable an explicit,
realistic finite bound and record assumptions.
"""
if live_api_ready:
    pareto = qqa.ask(
        pareto_request,
        solver="auto",
        sol_size=256,
        num_epochs=1000,
        device="auto",
    )
    display(pareto.plan.to_dict()["routing"])
    qqa.plot_pareto(pareto.result)
    qqa.plot_pareto_diagnostics(pareto.result)
else:
    print("Set the QQA_LLM_* API profile to run this live translation.")

## 4. Natural language → budget-aware black-box optimisation

Mentioning an expensive simulator or black-box experiment makes `auto` select batch surrogate optimisation. The safe expression is evaluated point by point, with no gradients exposed to the optimiser.

In [ ]:
blackbox_request = """
Treat reactor tuning as an expensive black-box experiment. Choose an
integer reactor count from 1 to 8 and real temperature from 300 to 500.
Minimize (reactors-4)^2 + ((temperature-410)/30)^2 with at most 96
parallelizable evaluations, subject to reactors*temperature <= 2800.
"""
if live_api_ready:
    blackbox = qqa.ask(
        blackbox_request,
        solver="auto",
        budget=96,
        batch_size=8,
        workers=8,
        device="auto",
    )
    display(blackbox.plan.to_dict()["routing"])
    display(blackbox.result.best_point)
    qqa.plot_blackbox(blackbox.result)
else:
    print("Set the QQA_LLM_* API profile to run this live translation.")

## CLI equivalents

The provider-neutral profile is read from `QQA_LLM_API_KEY`, `QQA_LLM_BASE_URL`, and `QQA_LLM_MODEL`. The key never appears in the command line, generated model, result JSON, or report.

In [ ]:
print("""qqa ask "Minimize (x-2)^2 for real x in [-5,5]" --plan-only --show-model
qqa ask --file realistic-request.txt --solver auto --device auto \\
  --output-plan plan.json --output-result result.json --report result.html
qqa ask --spec plan-model.json --solver qqa --device auto
qqa gui  # open the Ask QQA tab""")